In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IPL.csv")

C:\Users\Dwitiyash\AppData\Local\Temp\ipykernel_6712\1393846161.py:1: DtypeWarning: Columns (0: review_batter, 1: team_reviewed, 2: review_decision, 3: umpire, 4: season, 5: superover_winner, 6: result_type, 7: method, 8: event_match_no) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("IPL.csv")


In [4]:
df.describe()

,Unnamed: 0,match_id,innings,over,ball,ball_no,bat_pos,runs_batter,balls_faced,valid_ball,...,year,balls_per_over,overs,team_runs,team_balls,team_wicket,power_surge_start,batter_runs,batter_balls,bowler_wicket
count,283678.000000,2.836780e+05,283678.000000,283678.000000,283678.000000,283678.000000,283678.000000,283678.000000,283678.000000,283678.000000,...,283678.000000,283678.0,283678.0,283678.000000,283678.000000,283678.000000,0.0,283678.000000,283678.000000,283678.000000
mean,141838.500000,9.535684e+05,1.482787,9.190833,3.488847,9.539717,3.616445,1.281421,0.967082,0.962912,...,2016.889406,6.0,20.0,77.423099,58.596328,2.460624,NaN,18.376621,13.993813,0.045530
std,81890.929169,3.865091e+05,0.502511,5.680845,1.708286,5.682271,2.170569,1.655126,0.178421,0.188977,...,5.352468,0.0,0.0,50.167985,34.113517,2.102515,NaN,18.607095,11.815576,0.208465
min,0.000000,3.359820e+05,1.000000,0.000000,1.000000,0.100000,1.000000,0.000000,0.000000,0.000000,...,2008.000000,6.0,20.0,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000
25%,70919.250000,5.483590e+05,1.000000,4.000000,2.000000,4.500000,2.000000,0.000000,1.000000,1.000000,...,2012.000000,6.0,20.0,36.000000,29.000000,1.000000,NaN,4.000000,5.000000,0.000000
50%,141838.500000,1.082612e+06,1.000000,9.000000,3.000000,9.400000,3.000000,1.000000,1.000000,1.000000,...,2017.000000,6.0,20.0,73.000000,58.000000,2.000000,NaN,12.000000,11.000000,0.000000
75%,212757.750000,1.304066e+06,2.000000,14.000000,5.000000,14.400000,5.000000,1.000000,1.000000,1.000000,...,2022.000000,6.0,20.0,114.000000,88.000000,4.000000,NaN,27.000000,20.000000,0.000000
max,283677.000000,1.529267e+06,6.000000,19.000000,7.000000,19.600000,11.000000,6.000000,1.000000,1.000000,...,2026.000000,6.0,20.0,287.000000,121.000000,10.000000,NaN,175.000000,73.000000,1.000000


In [6]:
df.columns

Index(['Unnamed: 0', 'match_id', 'date', 'match_type', 'event_name', 'innings',
       'batting_team', 'bowling_team', 'over', 'ball', 'ball_no', 'batter',
       'bat_pos', 'runs_batter', 'balls_faced', 'bowler', 'valid_ball',
       'runs_extras', 'runs_total', 'runs_bowler', 'runs_not_boundary',
       'extra_type', 'non_striker', 'non_striker_pos', 'wicket_kind',
       'player_out', 'fielders', 'runs_target', 'review_batter',
       'team_reviewed', 'review_decision', 'umpire', 'umpires_call',
       'player_of_match', 'match_won_by', 'win_outcome', 'toss_winner',
       'toss_decision', 'venue', 'city', 'day', 'month', 'year', 'season',
       'gender', 'team_type', 'superover_winner', 'result_type', 'method',
       'balls_per_over', 'overs', 'event_match_no', 'stage', 'match_number',
       'team_runs', 'team_balls', 'team_wicket', 'new_batter',
       'power_surge_start', 'batter_runs', 'batter_balls', 'bowler_wicket',
       'batting_partners', 'next_batter', 'striker_out'],


In [7]:
df_5yr = df[df["year"].between(2021, 2025)].copy()

In [9]:
# Create one row per batter per match innings
innings = (
    df_5yr[["batter", "match_id", "innings"]]
    .drop_duplicates()
    .groupby("batter")
    .size()
    .reset_index(name="innings")
)

innings.head()

,batter,innings
0,A Badoni,46
1,A Kamboj,6
2,A Manohar,20
3,A Mhatre,7
4,A Mishra,1


In [10]:
outs = (
    df_5yr[
        (df_5yr["player_out"].notna()) &
        (df_5yr["striker_out"] == 1)
    ]
    .groupby("player_out")
    .size()
    .reset_index(name="outs")
    .rename(columns={"player_out": "batter"})
)

outs.head()

,batter,outs
0,A Badoni,35
1,A Kamboj,2
2,A Manohar,19
3,A Mhatre,7
4,A Mishra,1


In [11]:
batting_avg = innings.merge(
    outs,
    on="batter",
    how="left"
)

batting_avg["outs"] = batting_avg["outs"].fillna(0).astype(int)

batting_avg.head()

,batter,innings,outs
0,A Badoni,46,35
1,A Kamboj,6,2
2,A Manohar,20,19
3,A Mhatre,7,7
4,A Mishra,1,1


In [12]:
batting_5year=pd.read_csv("Batting_5year.csv")
batting_avg = batting_avg.merge(
    batting_5year[["batter", "runs_5yr"]],
    on="batter",
    how="left"
)

batting_avg.head()

,batter,innings,outs,runs_5yr
0,A Badoni,46,35,963
1,A Kamboj,6,2,16
2,A Manohar,20,19,292
3,A Mhatre,7,7,240
4,A Mishra,1,1,19


In [13]:
batting_avg["not_outs"] = (
    batting_avg["innings"] -
    batting_avg["outs"]
)

In [14]:
import numpy as np
batting_avg["batting_average"] = np.where(
    batting_avg["outs"] == 0,
    batting_avg["runs_5yr"],
    batting_avg["runs_5yr"] / batting_avg["outs"]
)

batting_avg["batting_average"] = (
    batting_avg["batting_average"]
    .round(2)
)

In [15]:
batting_avg = batting_avg[
    [
        "batter",
        "innings",
        "outs",
        "not_outs",
        "batting_average"
    ]
]

In [16]:
batting_avg.isnull().sum()

batter             0
innings            0
outs               0
not_outs           0
batting_average    0
dtype: int64

In [17]:
batting_avg["batter"].duplicated().sum()

np.int64(0)

In [18]:
batting_avg.sort_values(
    "batting_average",
    ascending=False
).head(10)

,batter,innings,outs,not_outs,batting_average
321,Vivrant Sharma,1,1,0,69.00
274,SS Tiwary,4,2,2,57.50
43,B Sai Sudharsan,40,35,5,51.23
128,KL Rahul,64,51,13,50.63
50,CA Lynn,1,1,0,49.00
297,T Stubbs,31,15,16,47.40
85,H Klaasen,39,31,8,45.61
70,DP Conway,28,24,4,45.00
103,JC Buttler,62,54,8,44.56
311,V Kohli,75,64,11,43.48


In [19]:
print("Players:", batting_avg.shape[0])
print("Total Innings:", batting_avg["innings"].sum())
print("Total Outs:", batting_avg["outs"].sum())
print("Average Batting Average:", batting_avg["batting_average"].mean().round(2))

Players: 332
Total Innings: 5484
Total Outs: 4172
Average Batting Average: 17.45


In [20]:
batting_avg.to_csv("batting_average.csv", index=False)